In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/monday[1].csv",
    "/content/drive/MyDrive/thursday[1].csv",
    "/content/drive/MyDrive/monday_plus[1].csv",
    "/content/drive/MyDrive/tuesday[1].csv",
    "/content/drive/MyDrive/thursday_plus[1].csv",
    "/content/drive/MyDrive/tuesday_plus[1].csv",
    "/content/drive/MyDrive/wednesday[1].csv",
    "/content/drive/MyDrive/friday[1].csv",
    "/content/drive/MyDrive/friday_plus[1].csv",
    "/content/drive/MyDrive/wednesday_plus[1].csv"
]

dfs = []

for f in files:
    dfs.append(pd.read_csv(f, nrows=50000))  # load only 50k rows per CSV

# Combine all samples
df = pd.concat(dfs, ignore_index=True)
print("Combined dataset shape:", df.shape)
df.head()


Combined dataset shape: (500000, 105)


,Src IP dec,Src Port,Dst IP dec,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,...,Local_5,Local_6,Local_7,Local_8,Local_9,Local_10,Local_11,Local_12,Local_13,Local_14
0,134610945,0,134219268,0,0,56:34.2,119719148,231,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3232238089,123,3232238083,123,17,56:55.4,65511209,6,6,288,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3232238092,5353,3758096635,5353,17,57:21.1,113976922,267,0,20447,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3232238092,123,2550302004,123,17,57:31.6,67037196,8,8,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3232238092,123,760155097,123,17,57:30.6,68045057,8,8,384,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import numpy as np

df = df.dropna()                      # remove missing values
df = df.replace([np.inf, -np.inf], 0) # replace infinite values


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])  # BENIGN=0, ATTACK=1


In [ ]:
X = df.drop("Label", axis=1)
y = df["Label"]


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Keep only numeric columns
X_numeric = X.select_dtypes(include='number')

print("Columns used for scaling:", X_numeric.columns)

# Scale numeric features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_numeric)

X = X_scaled  # now X is numeric and safe for ANN/CNN



Columns used for scaling: Index(['Src IP dec', 'Src Port', 'Dst IP dec', 'Dst Port', 'Protocol',
       'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       ...
       'Local_5', 'Local_6', 'Local_7', 'Local_8', 'Local_9', 'Local_10',
       'Local_11', 'Local_12', 'Local_13', 'Local_14'],
      dtype='object', length=102)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Compute class weights
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = dict(enumerate(class_weights))


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten, Dropout, Reshape

# Reshape input for Conv1D: (samples, features, 1)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

model_cnn = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1],1)),
    Dropout(0.3),
    Conv1D(32, kernel_size=3, activation='relu'),
    Dropout(0.2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(len(np.unique(y_train)), activation='softmax')  # multiclass
])

model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model_cnn.fit(
    X_train_cnn, y_train,
    epochs=30, batch_size=64,
    validation_split=0.2,
    class_weight=class_weights_dict
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - accuracy: 0.0548 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 2/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 3/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 4/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 5/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 6/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 7/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.0000e+00 - loss: nan - val_accuracy: 0.0000e+00 - val_loss: nan
Epoch 8/30
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.0000e+00

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = model_cnn.predict(X_test_cnn).argmax(axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Confusion Matrix:\n", cm)


1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step
Accuracy: 0.88638
Precision: 0.7856695044
Recall: 0.88638
F1-score: 0.8329917666641927
Confusion Matrix:
 [[44319     0     0     0     0     0     0     0     0     0     0]
 [  255     0     0     0     0     0     0     0     0     0     0]
 [ 3181     0     0     0     0     0     0     0     0     0     0]
 [    1     0     0     0     0     0     0     0     0     0     0]
 [  346     0     0     0     0     0     0     0     0     0     0]
 [  688     0     0     0     0     0     0     0     0     0     0]
 [  813     0     0     0     0     0     0     0     0     0     0]
 [  386     0     0     0     0     0     0     0     0     0     0]
 [    2     0     0     0     0     0     0     0     0     0     0]
 [    1     0     0     0     0     0     0     0     0     0     0]
 [    8     0     0     0     0     0     0     0     0     0     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
